# 🌉 Bridge Damage Detection — Benchmark Notebook (FIXED)
**Dataset:** dacl10k (Supervisely format) | **Task:** Semantic Segmentation | **Classes:** 19

| Step | Deskripsi |
|---|---|
| 1 | Cek GPU + Mount Google Drive |
| 2 | Install dependencies |
| 3 | Verifikasi dataset |
| 4 | Visualisasi sampel |
| 5 | Konfigurasi & DataLoader |
| 6 | Load model |
| 7 | Training |
| 8 | Plot training curves |
| 9 | Evaluasi test set |
| 10 | Robustness evaluation |
| 11 | Visualisasi prediksi |
| 12 | Resume dari checkpoint |

> **Sebelum mulai:** Runtime → Change runtime type → **GPU (A100 / T4)**

## ✅ Step 1 — Cek GPU & Mount Google Drive

In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  Tidak ada GPU — ganti runtime ke GPU!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ⚠️  SESUAIKAN path ini dengan lokasi project di Drive Anda
PROJECT_PATH = '/content/drive/MyDrive/Penelitian/bridge-benchmark-cv'

if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    # Tambahkan ke sys.path agar semua import bisa ditemukan
    if PROJECT_PATH not in sys.path:
        sys.path.insert(0, PROJECT_PATH)
    print(f'✅ Working directory: {os.getcwd()}')
    print('Isi folder:')
    for item in sorted(os.listdir('.')):
        print(f'  {item}')
else:
    print(f'❌ Path tidak ditemukan: {PROJECT_PATH}')
    print('Sesuaikan PROJECT_PATH di cell ini!')

## 📦 Step 2 — Install Dependencies

In [ ]:
# Install semua package yang dibutuhkan
# torch sudah ada di Colab — kita install sisanya

!pip install -q "transformers>=4.41.0"
!pip install -q timm segmentation-models-pytorch
!pip install -q "albumentations==1.4.3"
!pip install -q "torchmetrics>=1.3.0"
!pip install -q einops pycocotools

print('\n✅ Semua dependencies terinstall!')

In [ ]:
# Verifikasi semua import
import torch, torchvision, albumentations, transformers, torchmetrics
import segmentation_models_pytorch as smp
import timm

print(f'torch          : {torch.__version__}')
print(f'transformers   : {transformers.__version__}')
print(f'timm           : {timm.__version__}')
print(f'smp            : {smp.__version__}')
print(f'albumentations : {albumentations.__version__}')
print(f'torchmetrics   : {torchmetrics.__version__}')
print('\n✅ Semua import OK!')

In [ ]:
# Pastikan __init__.py ada di semua subfolder
import os
for folder in ['train', 'datasets', 'models', 'eval']:
    init = os.path.join(folder, '__init__.py')
    os.makedirs(folder, exist_ok=True)
    if not os.path.exists(init):
        open(init, 'w').write('')
        print(f'Created: {init}')
    else:
        print(f'OK     : {init}')

## 🗂️ Step 3 — Verifikasi Dataset

In [ ]:
from pathlib import Path

DATA_ROOT = 'data/dacl10k'

print('=== Verifikasi Struktur Dataset ===')
total_imgs = 0
for split in ['train', 'val', 'test']:
    img_dir = Path(DATA_ROOT) / split / 'img'
    ann_dir = Path(DATA_ROOT) / split / 'ann'
    imgs = list(img_dir.glob('*.*')) if img_dir.exists() else []
    anns = list(ann_dir.glob('*.json')) if ann_dir.exists() else []
    status = '✅' if len(imgs) > 0 else '❌'
    total_imgs += len(imgs)
    print(f'  {status} [{split:5s}] {len(imgs):5d} gambar | {len(anns):5d} JSON')

print(f'\nTotal: {total_imgs} gambar')
if total_imgs == 0:
    print('\n⚠️  Dataset kosong! Pastikan dacl10k sudah di-download.')

In [ ]:
# Import loader menggunakan importlib (cara paling robust di Colab)
import importlib.util, sys, os

def import_from_file(module_name, file_path):
    """Load module langsung dari path file — bypass sys.path issues."""
    if module_name in sys.modules:
        del sys.modules[module_name]
    spec   = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

# Load dacl10k_loader
loader_mod = import_from_file(
    'datasets.dacl10k_loader',
    'datasets/dacl10k_loader.py'
)

Dacl10kDataset = loader_mod.Dacl10kDataset
build_dataloaders = loader_mod.build_dataloaders
build_robust_loader = loader_mod.build_robust_loader
ALL_CLASSES = loader_mod.ALL_CLASSES
NUM_CLASSES = loader_mod.NUM_CLASSES

# Test load 1 sampel
ds_test = Dacl10kDataset(root_dir=DATA_ROOT, split='train', debug=True)
sample = ds_test[0]
print(f'\nSample [0]:')
print(f'  image  : {sample["image"].shape}  {sample["image"].dtype}')
print(f'  mask   : {sample["mask"].shape}   {sample["mask"].dtype}')
print(f'  labels : {sample["mask"].unique().tolist()}')
print(f'  id     : {sample["image_id"]}')
print(f'\n✅ DataLoader OK! ({NUM_CLASSES} kelas)')

## 🖼️ Step 4 — Visualisasi Sampel

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CLASS_COLORS = [
    (255,192,203),(178,223,138),(0,255,128),(0,0,255),(139,0,0),
    (255,140,0),(169,169,169),(207,52,118),(128,0,0),(246,124,104),
    (128,0,128),(128,128,0),(212,175,162),(253,191,111),(255,0,0),
    (123,211,247),(0,0,139),(0,128,0),(0,139,139),
]

def denormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.permute(1,2,0).numpy()
    return (img * std + mean).clip(0,1)

def mask_to_rgb(mask_tensor):
    mask = mask_tensor.numpy()
    rgb  = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for idx, color in enumerate(CLASS_COLORS):
        rgb[mask == idx] = color
    rgb[mask == 255] = (50, 50, 50)
    return rgb

os.makedirs('outputs/figures', exist_ok=True)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('dacl10k — Sample Gambar & Mask', fontsize=14, fontweight='bold')
for i in range(4):
    s = ds_test[i]
    axes[0,i].imshow(denormalize(s['image']))
    axes[0,i].set_title(f'Image [{i}]', fontsize=9)
    axes[0,i].axis('off')
    axes[1,i].imshow(mask_to_rgb(s['mask']))
    axes[1,i].set_title(f'Mask [{i}]', fontsize=9)
    axes[1,i].axis('off')

plt.tight_layout()
plt.savefig('outputs/figures/sample_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisasi tersimpan di outputs/figures/')

## ⚙️ Step 5 — Konfigurasi & DataLoader

> **Ganti `MODEL_NAME`** untuk training model berbeda:
> `'segformer'` · `'deeplabv3'` · `'unet_effnet'` · `'unet_resnet'`

In [ ]:
# ── KONFIGURASI UTAMA ──────────────────────────────────────────────
MODEL_NAME  = 'segformer'   # ← GANTI DI SINI untuk model berbeda
DATA_ROOT   = 'data/dacl10k'
IMG_SIZE    = 512
BATCH_SIZE  = 8             # Kurangi ke 4 jika OOM (Out of Memory)
NUM_WORKERS = 2
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

CONFIG = {
    'model_name'    : MODEL_NAME,
    'epochs'        : 50,
    'lr'            : 1e-4,
    'weight_decay'  : 1e-4,
    'loss_alpha'    : 0.5,
    'use_focal'     : False,
    'scheduler'     : 'cosine',
    'warmup_epochs' : 3,
    'patience'      : 10,
    'use_amp'       : True,
    'num_classes'   : 19,
    'ignore_index'  : 255,
    'img_size'      : IMG_SIZE,
    'output_dir'    : 'outputs',
}

print(f'Model    : {MODEL_NAME}')
print(f'Device   : {DEVICE}')
print(f'Batch    : {BATCH_SIZE}')
print(f'Config   : {CONFIG}')

# Build DataLoaders
train_loader, val_loader, test_loader = build_dataloaders(
    root_dir    = DATA_ROOT,
    img_size    = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
)
print(f'\nTrain batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')
print('\n✅ DataLoaders siap!')

## 🤖 Step 6 — Load Model

In [ ]:
def load_model(model_name: str, num_classes: int = 19):
    """Factory function: load model berdasarkan nama."""
    import segmentation_models_pytorch as smp
    from transformers import SegformerForSemanticSegmentation

    if model_name == 'segformer':
        print('Loading SegFormer-B2 (Transformer)...')
        model = SegformerForSemanticSegmentation.from_pretrained(
            'nvidia/segformer-b2-finetuned-ade-512-512',
            num_labels=num_classes,
            ignore_mismatched_sizes=True,
        )
    elif model_name == 'deeplabv3':
        print('Loading DeepLabV3+ (ResNet50)...')
        model = smp.DeepLabV3Plus(
            encoder_name='resnet50', encoder_weights='imagenet',
            in_channels=3, classes=num_classes,
        )
    elif model_name == 'unet_effnet':
        print('Loading UNet-EfficientNetB4...')
        model = smp.Unet(
            encoder_name='efficientnet-b4', encoder_weights='imagenet',
            in_channels=3, classes=num_classes,
        )
    elif model_name == 'unet_resnet':
        print('Loading UNet-ResNet101...')
        model = smp.Unet(
            encoder_name='resnet101', encoder_weights='imagenet',
            in_channels=3, classes=num_classes,
        )
    else:
        raise ValueError(f'Model tidak dikenal: {model_name}')

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'✅ Model loaded | Parameters: {n_params:.1f}M')
    return model

model = load_model(MODEL_NAME, num_classes=19)

## 🏋️ Step 7 — Training

In [ ]:
import importlib.util, sys

def import_from_file(module_name, file_path):
    """Load module langsung dari path — bypass sys.path issues."""
    if module_name in sys.modules:
        del sys.modules[module_name]
    spec   = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

# Load losses & trainer
losses_mod  = import_from_file('train.losses',  'train/losses.py')
trainer_mod = import_from_file('train.trainer', 'train/trainer.py')
Trainer     = trainer_mod.Trainer

print('✅ Trainer berhasil di-import!')

# Buat folder output
os.makedirs(f'outputs/{MODEL_NAME}', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)

# Inisialisasi trainer
trainer = Trainer(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    config       = CONFIG,
    device       = DEVICE,
)

# Mulai training!
best_miou = trainer.fit()
print(f'\n🏆 Training selesai! Best val mIoU = {best_miou:.4f}')

## 📈 Step 8 — Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = f'outputs/{MODEL_NAME}/training_log.csv'
df = pd.read_csv(log_path)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Training Curves — {MODEL_NAME}', fontsize=13, fontweight='bold')

axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss', color='#3b82f6')
axes[0].plot(df['epoch'], df['val_loss'],   label='Val Loss',   color='#ef4444', linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df['epoch'], df['val_miou'], label='Val mIoU', color='#10b981', linewidth=2)
best_epoch = df['val_miou'].idxmax()
axes[1].axvline(df['epoch'][best_epoch], color='orange', linestyle=':',
               label=f'Best epoch={df["epoch"][best_epoch]}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
axes[1].set_title('Validation mIoU'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'outputs/figures/{MODEL_NAME}_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best mIoU : {df["val_miou"].max():.4f} (epoch {df["epoch"][best_epoch]})')

## 📊 Step 9 — Evaluasi Test Set

In [ ]:
from torchmetrics import JaccardIndex
from tqdm.notebook import tqdm

# Load best model
ckpt = torch.load(f'outputs/{MODEL_NAME}/best_model.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model = model.to(DEVICE)
model.eval()
print(f'Loaded best model — epoch {ckpt["epoch"]} | mIoU={ckpt["best_miou"]:.4f}')

metric = JaccardIndex(
    task='multiclass', num_classes=19,
    average='none', ignore_index=255
).to(DEVICE)

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Evaluasi test set'):
        images  = batch['image'].to(DEVICE)
        targets = batch['mask'].to(DEVICE)
        logits  = model(images)
        if isinstance(logits, dict):
            logits = logits.get('logits', logits.get('out'))
        if logits.shape[-2:] != targets.shape[-2:]:
            logits = torch.nn.functional.interpolate(
                logits, size=targets.shape[-2:],
                mode='bilinear', align_corners=False)
        metric.update(logits.argmax(1), targets)

miou_per_class = metric.compute().cpu()
miou_mean      = miou_per_class.mean().item()

print(f'\n=== Hasil Evaluasi: {MODEL_NAME} ===')
print(f'{"Idx":<4} {"Class":<32} {"IoU":>8}')
print('-' * 46)
for i, (cls, iou) in enumerate(zip(ALL_CLASSES, miou_per_class)):
    bar = '█' * int(iou.item() * 20)
    print(f'[{i:2d}] {cls:<32} {iou.item():>6.4f}  {bar}')
print('-' * 46)
print(f'     {"MEAN mIoU":<32} {miou_mean:>6.4f}')

In [ ]:
# Simpan hasil ke CSV
import pandas as pd

os.makedirs('outputs/results', exist_ok=True)
results_df = pd.DataFrame({
    'class' : ALL_CLASSES + ['MEAN'],
    'iou'   : miou_per_class.tolist() + [miou_mean],
    'model' : MODEL_NAME,
})
results_path = f'outputs/results/{MODEL_NAME}_test_results.csv'
results_df.to_csv(results_path, index=False)
print(f'✅ Hasil tersimpan: {results_path}')
print(f'🏆 {MODEL_NAME} Test mIoU = {miou_mean:.4f}')

## 🌫️ Step 10 — Robustness Evaluation

In [ ]:
robust_results = {'normal': miou_mean}

for corruption in ['blur', 'noise', 'brightness']:
    robust_loader = build_robust_loader(
        root_dir   = DATA_ROOT,
        corruption = corruption,
        batch_size = BATCH_SIZE,
        num_workers= NUM_WORKERS,
    )
    metric.reset()
    with torch.no_grad():
        for batch in tqdm(robust_loader, desc=f'Robust [{corruption}]'):
            images  = batch['image'].to(DEVICE)
            targets = batch['mask'].to(DEVICE)
            logits  = model(images)
            if isinstance(logits, dict):
                logits = logits.get('logits', logits.get('out'))
            if logits.shape[-2:] != targets.shape[-2:]:
                logits = torch.nn.functional.interpolate(
                    logits, size=targets.shape[-2:],
                    mode='bilinear', align_corners=False)
            metric.update(logits.argmax(1), targets)

    rob_miou = metric.compute().mean().item()
    delta    = miou_mean - rob_miou
    robust_results[corruption] = rob_miou
    print(f'  [{corruption:<12}] mIoU={rob_miou:.4f}  ΔmIoU={delta:+.4f}')

print(f'\n=== Robustness Summary: {MODEL_NAME} ===')
for k, v in robust_results.items():
    delta = (miou_mean - v) if k != 'normal' else 0.0
    print(f'  {k:<12} : {v:.4f}  (Δ={delta:+.4f})')

## 🖼️ Step 11 — Visualisasi Prediksi

In [ ]:
model.eval()
batch   = next(iter(test_loader))
images  = batch['image'][:4].to(DEVICE)
targets = batch['mask'][:4]

with torch.no_grad():
    logits = model(images)
    if isinstance(logits, dict):
        logits = logits.get('logits', logits.get('out'))
    if logits.shape[-2:] != targets.shape[-2:]:
        logits = torch.nn.functional.interpolate(
            logits, size=targets.shape[-2:],
            mode='bilinear', align_corners=False)
    preds = logits.argmax(1).cpu()

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle(f'Prediksi Segmentasi — {MODEL_NAME}', fontsize=13, fontweight='bold')

for col in range(4):
    axes[0, col].imshow(denormalize(batch['image'][col]))
    axes[0, col].set_title(f'Sample {col}', fontsize=9)
    axes[1, col].imshow(mask_to_rgb(targets[col]))
    axes[2, col].imshow(mask_to_rgb(preds[col]))
    for row in range(3):
        axes[row, col].axis('off')

for row, label in enumerate(['Input Image', 'Ground Truth', 'Prediction']):
    axes[row, 0].set_ylabel(label, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f'outputs/figures/{MODEL_NAME}_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Tersimpan: outputs/figures/{MODEL_NAME}_predictions.png')

## 🔄 Step 12 — Resume dari Checkpoint

> Jalankan cell ini **jika session terputus** dan ingin melanjutkan training yang belum selesai.
> **Jangan jalankan** jika training sudah selesai di Step 7.

In [ ]:
# ── RESUME TRAINING ────────────────────────────────────────────────
# Pilih checkpoint yang ingin di-resume:
#   last_checkpoint.pth  → lanjut dari epoch terakhir
#   best_model.pth       → lanjut dari epoch terbaik

RESUME_FROM = f'outputs/{MODEL_NAME}/last_checkpoint.pth'

if not os.path.exists(RESUME_FROM):
    print(f'❌ Checkpoint tidak ditemukan: {RESUME_FROM}')
else:
    # Load checkpoint
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
    last_epoch = ckpt['epoch']
    print(f'Checkpoint ditemukan: epoch={last_epoch} | mIoU={ckpt["best_miou"]:.4f}')

    # Re-inisialisasi trainer
    trainer_resume = Trainer(
        model        = model,
        train_loader = train_loader,
        val_loader   = val_loader,
        config       = CONFIG,
        device       = DEVICE,
    )

    # Load weights & optimizer state
    start_epoch = trainer_resume.load_checkpoint(RESUME_FROM)

    # Fast-forward scheduler ke posisi yang benar
    for _ in range(last_epoch):
        trainer_resume.scheduler.step()
    print(f'Scheduler fast-forward ke epoch {last_epoch}')

    # Lanjutkan training dari epoch berikutnya
    print(f'\nMelanjutkan training dari epoch {start_epoch}...')
    best_miou = trainer_resume.fit(start_epoch=start_epoch)
    print(f'\n🏆 Training selesai! Best val mIoU = {best_miou:.4f}')

## 📋 Summary & Checklist Model

Setelah satu model selesai, ganti `MODEL_NAME` di **Step 5** dan jalankan ulang dari **Step 6**.

| Model | Status | Best mIoU |
|---|---|---|
| `segformer` | ⏳ | — |
| `deeplabv3` | ⏳ | — |
| `unet_effnet` | ⏳ | — |
| `unet_resnet` | ⏳ | — |

> Update tabel ini secara manual setelah setiap model selesai.